## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-27-13-50-49 +0000,wapo,"Power outages, heavy rain continue on East Coa...",https://www.washingtonpost.com/nation/2026/11/...
1,2026-11-27-13-50-15 +0000,bbc,What we know about Department for Work and Pen...,https://www.bbc.co.uk/news/articles/ck1l0e946d...
2,2026-11-27-13-46-00 +0000,wsj,Stock Futures Jump as AI Trade Set for Rebound,https://www.wsj.com/finance/stocks/stock-futur...
3,2026-11-27-13-45-52 +0000,cbc,Supreme Court set to deliver ruling on Quebec'...,https://www.cbc.ca/news/canada/montreal/livest...
4,2026-11-27-13-40-20 +0000,nyt,Photos: Macy’s Thanksgiving Day Parade History...,https://www.nytimes.com/2026/11/26/nyregion/ma...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2632/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
7,thanksgiving,23
99,trump,21
129,he,14
121,new,13
17,ai,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
267,2026-11-26-15-39-23 +0000,nypost,Trump tells Putin to ‘pull back all military a...,https://nypost.com/2026/11/26/us-news/trump-te...,70
0,2026-11-27-13-50-49 +0000,wapo,"Power outages, heavy rain continue on East Coa...",https://www.washingtonpost.com/nation/2026/11/...,69
197,2026-11-26-20-37-19 +0000,nypost,Trump celebrates Thanksgiving with his holiday...,https://nypost.com/2026/11/26/us-news/trump-ce...,68
91,2026-11-27-09-57-50 +0000,cbc,Thanksgiving storm on U.S. East Coast knocks o...,https://www.cbc.ca/news/world/u-s-thanksgiving...,66
29,2026-11-27-12-24-03 +0000,chitri,Ukraine president urges US to pressure Vladimi...,https://www.chicagotribune.com/2026/11/27/ukra...,62


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
267,70,2026-11-26-15-39-23 +0000,nypost,Trump tells Putin to ‘pull back all military a...,https://nypost.com/2026/11/26/us-news/trump-te...
0,69,2026-11-27-13-50-49 +0000,wapo,"Power outages, heavy rain continue on East Coa...",https://www.washingtonpost.com/nation/2026/11/...
135,54,2026-11-27-04-06-00 +0000,wsj,New Zealand’s election ended in a deadlock aft...,https://www.wsj.com/politics/elections/new-zea...
245,44,2026-11-26-17-12-53 +0000,wapo,Two Italians killed in Kuwait as a strained Ir...,https://www.washingtonpost.com/world/2026/11/2...
229,35,2026-11-26-18-14-09 +0000,nypost,Six in 10 shoppers plan to use AI for Black Fr...,https://nypost.com/2026/11/26/business/six-in-...
129,34,2026-11-27-05-07-41 +0000,nypost,"UK shields fighter jets, contacts reservists i...",https://nypost.com/2026/11/27/world-news/uk-sh...
30,34,2026-11-27-12-18-10 +0000,nypost,Turning Point USA mourns Fla. high school chap...,https://nypost.com/2026/11/27/us-news/turning-...
55,33,2026-11-27-11-00-00 +0000,nypost,Inside alleged pedophile Thomas Markle’s parad...,https://nypost.com/2026/11/27/world-news/insid...
22,29,2026-11-27-12-51-00 +0000,wsj,"Week Ahead for FX, Bonds: Key Labor Data in Fo...",https://www.wsj.com/finance/investing/week-ahe...
174,28,2026-11-26-23-03-16 +0000,bbc,My dad's watch stopped the moment he died - I ...,https://www.bbc.co.uk/news/videos/c6evr2y40v0j...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
